In [1]:
# PRL thickness polar maps from arivis Cloud segmentation masks

# Expected input layout (masks: 0 = background, 255 = foreground):
#   data/
#   └── <condition>_C/                      e.g. WT5dpf_C
#       ├── <condition>_raw/<fish_ID>/       raw micro-CT slices (.tiff); one folder per fish
#       ├── <condition>_cleaned_ONL/<fish_ID>_ONL/    PRL mask, 360 rotational slices
#       └── <condition>_cleaned_lens/<fish_ID>_lens/  lens mask (first slice is used)

# The model class "ONL" is the photoreceptor layer (PRL): cell bodies, inner and outer segments.
# Voxel size is 1 um (isotropic), so thickness values are in um.

# Outputs are written to outputs/:
#   outputs/data/<fish_ID>.npy   per-fish 360 x 180 thickness map
#   outputs/img/<fish_ID>.png    per-fish QC figure
#   outputs/polar/<group>.png    group-mean polar map

from scipy.signal import find_peaks
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import skimage.io as skio
import seaborn as sns
import plotly.express as px
import cv2
from skimage import morphology
from scipy.ndimage import gaussian_filter
from scipy.ndimage import binary_erosion,binary_closing
import matplotlib.colors as mcolors

#  Paths: edit these to point to your data and output folders ----
DATA_DIR = ""       # folder containing the <condition>_C folders
OUTPUT_DIR =""   # results are written here

# Eye-centre method 
# "lens": centre row from the lens mask (average vertical midpoint of the lens, measured
#         column by column from the left edge). Needs only the masks.
# "raw":  centre row from the raw micro-CT image: midpoint between the two highest
#         intensity peaks along the left-edge column of the first raw slice.
#         Needs the raw .tiff files in <condition>_raw/<fish_ID>/.

CENTRE_METHOD = "lens"

# Group-level polar map: contour map of PRL thickness plus two rings of 10-degree sector bars
# (inner ring: mean thickness; outer ring: 99th-percentile thickness).
# Values below vmin (um) are set to NaN and excluded from the map and the sector bars.

def plot_individual_layer(data_to_plot,title, save = False, save_dir = None):
    vmin, vmax = 5, 25   # colour-scale limits (um)
    
    radius = np.linspace(0, 120, 121)  
    theta = np.linspace(0, 2*np.pi, 360) 
    data_to_plot[data_to_plot < vmin] = np.nan
    T, R = np.meshgrid(theta, radius)
    
    # plt.figure()
    fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
    data_to_plot = gaussian_filter(data_to_plot, sigma = 0)

    c = ax.contourf(T, R, data_to_plot.T[:121], levels = 100, cmap='viridis',vmin = vmin, vmax = vmax,)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    # plt.colorbar(c, ax = ax)
    plt.title(f'{title}')

    degree_segments = 10

    # Create bins for the degrees (Z-coordinate)
    bins = np.arange(0, 360 + degree_segments, degree_segments)

    theta_edges = np.radians(bins)
    max_r = 120
    scaling_factor_ave = 0
    scaling_factor_max = 0.3
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap('viridis')           

    for i in range(len(theta_edges) - 1):
        arc_start = theta_edges[i]
        arc_end = theta_edges[i + 1]
        arc_thickness = np.nanmean(data_to_plot[10*i:10*i+10])
        
        # Position the bars outside the maximum radius
        r_bar_position = max_r + scaling_factor_ave * max_r
        
        # Create a bar
        ax.bar(
            (arc_start + arc_end) / 2,  # Position the bar in the middle of the segment
            10,  # Radial length of the bar scaled for visibility
            width=(arc_end - arc_start), # Angular width of the bar
            bottom=r_bar_position,       # Starting position outside the main plot
            color= cmap(norm(arc_thickness)),  # Color based on thickness
            edgecolor='black',           # Optional: add black edge to the bar
            alpha=0.8                    # Transparency
        )


    for i in range(len(theta_edges) - 1):
        arc_start = theta_edges[i]
        arc_end = theta_edges[i + 1]

        a = sorted(data_to_plot[10*i:10*i+10].flatten())
        cleanedList = [x for x in a if x == x]
        arc_max_thickness = cleanedList[int(0.99 * len(cleanedList))]
        
        # Position the bars further outside the average thickness bars
        r_bar_position_max = max_r + 0.1 * max_r
        
        # Create a bar for maximum thickness
        ax.bar(
            (arc_start + arc_end) / 2,  # Position the bar in the middle of the segment
            arc_max_thickness * scaling_factor_max * max_r / (vmax-vmin),  # Radial length of the bar scaled for visibility
            width=(arc_end - arc_start), # Angular width of the bar
            bottom=r_bar_position_max,   # Starting position outside the average thickness bars
            color= cmap(norm(arc_max_thickness)),  # Color based on thickness
            edgecolor='black',           # Add black edge to the bar for prominence
            alpha=0.8                    # Transparency
        )

    cbar = plt.colorbar(c, ax = ax)

    cbar_levels = np.arange(vmin, vmax+1, 2) 
    cbar.set_ticks(cbar_levels)
    plt.show()
    ax.set_rmax(175)

    if save == True:
        directory = os.path.join(OUTPUT_DIR, "polar", save_dir)
        dir_folder = os.path.dirname(directory)
        os.makedirs(dir_folder, exist_ok=True)
        fig.savefig(directory, dpi=300)

# Per-fish measurement: for each fish, estimate the eye centre from the lens mask, cast
# 180 rays (1-degree steps) from the eye centre on each of the 360 slices, and count PRL
# voxels along each ray. Saves a 360 x 180 thickness map (.npy) and a QC figure per fish.

def all_in_one(conditions):
    def find_two_highest_peaks(data):
        peaks, properties = find_peaks(data, height=0, distance = 50) 
        if len(peaks) == 0:
            return ()
        elif len(peaks) == 1:
            return (peaks[0],)
        else:
            peak_heights = properties['peak_heights']
            sorted_indices = np.argsort(peak_heights)[::-1]
            a, b = tuple(peaks[sorted_indices[:2]])
            return (a+b)/2


    for condition in conditions:
        print(f'Starting {condition}..')
        condition_dir = os.path.join(DATA_DIR, f"{condition}_C")
        # Fish IDs are taken from the folder names inside <condition>_raw
        paths = sorted(glob.glob(os.path.join(condition_dir, "*raw", "*")))
        ID_list = list(set([os.path.basename(os.path.normpath(p)) for p in paths]))
        
        for j in range(len(ID_list)):
            data_ONL = sorted(glob.glob(os.path.join(condition_dir, f"{condition}_cleaned_ONL", f"{ID_list[j]}*ONL", "*")))
            data_raw = sorted(glob.glob(os.path.join(condition_dir, f"{condition}_raw", ID_list[j], "*.tiff")))
            data_lens = sorted(glob.glob(os.path.join(condition_dir, f"{condition}_cleaned_lens", f"{ID_list[j]}*lens", "*")))

            # Load the 360 PRL mask slices; each slice is made portrait and padded/cropped to 350 x 200 px
            data = np.zeros((360,350,200))
            for idx, file_path in enumerate(data_ONL):
                temp_data = skio.imread(file_path)
            
                if temp_data.shape[0] < temp_data.shape[1]:
                    temp_data = temp_data.T
            
                h, w = temp_data.shape
                pad_h = max(0, 350 - h)
                pad_w = max(0, 200 - w)
            
                if pad_h > 0 or pad_w > 0:
                    temp_data = np.pad(temp_data, ((0, pad_h), (0, pad_w)), mode="constant")
            
                data[idx] = temp_data[:350, :200]

            # Eye centre (CoE, row coordinate); method set by CENTRE_METHOD above 
            if CENTRE_METHOD == "raw":
                # Midpoint between the two highest intensity peaks along the left-edge
                # column of the first raw slice (after .T, temp_data[0] is that column)
                temp_data = skio.imread(data_raw[0]).T
                CoE = find_two_highest_peaks(temp_data[0]) # Centre of eye all
                if isinstance(CoE, tuple):
                    raise ValueError(f"{ID_list[j]}: fewer than two intensity peaks found in the raw image; "
                                     "try CENTRE_METHOD = 'lens'")

            elif CENTRE_METHOD == "lens":
                # Mean row of lens pixels in each column, averaged over columns from the
                # left edge until the first column without lens (first lens slice only)
                mini_mean_list = []
                eye_data = skio.imread(data_lens[0]).T
                for eye_index in range(eye_data.shape[0]):
                    non_zero_pixels = np.where(eye_data[eye_index] == 255)[0]
                    # print(non_zero_pixels)
                    if non_zero_pixels.any() != False:
                        mini_mean_list.append(non_zero_pixels.mean())
                    else:
                        break
                CoE = int(np.array(mini_mean_list).mean())

            else:
                raise ValueError(f"CENTRE_METHOD must be 'lens' or 'raw', not {CENTRE_METHOD!r}")
            
            # Cast 180 rays (angles -90 to +89 degrees) of 200 samples from the eye centre on every slice
            x0 = 0 
            y0 = CoE
            Z = []

            for imstack in data:
                length = 200

                a = []
                for angle in range(180):
                    a.append([])

                    xrange = np.linspace(0, length * np.cos(np.deg2rad(angle-90)), length) + x0
                    yrange = np.linspace(0, length * np.sin(np.deg2rad(angle-90)), length) + y0

                    for i in range(length):
                        try:
                            xvalue, yvalue = int(xrange[i]), int(yrange[i])
                            if xvalue > 0:
                                if yvalue > 0:
                                    a[angle].append(imstack[yvalue][xvalue])
                                else:
                                    a[angle].append(0)
                            else:
                                a[angle].append(0)
                        except:
                            a[angle].append(0)
                a = np.array(a)
                Z.append(a)
            # Thickness = number of PRL voxels along each ray (1 voxel = 1 um)
            Z = np.array(Z)/255

            grand_thickness = []
            for theta in range(len(Z)):
                temp = Z[theta].sum(axis=1)
                temp = temp[::-1]

                for thickness in temp:
                    grand_thickness.append(thickness)

            grand_thickness = np.array(grand_thickness)
            grand_thickness = grand_thickness.reshape(360,180)

            # Per-fish QC figure: thickness map (left) and polar view (right)
            fig = plt.figure()
            ax1 = fig.add_subplot(1, 2, 1)
            ax1.imshow(grand_thickness,vmin = 0, vmax = 30)

            ax2 = fig.add_subplot(1, 2, 2, projection= 'polar')

            radius = np.linspace(0, 179, 180)  
            theta = np.linspace(0, 2*np.pi, 360) 
            T, R = np.meshgrid(theta, radius)

            c = ax2.contourf(T, R, grand_thickness.T, levels = 50, cmap='viridis',vmin = 0, vmax = 30)
            ax2.set_theta_zero_location("N")
            ax2.set_theta_direction(-1)
            ax2.set_rmax(180)

            plt.colorbar(c, ax = ax2,)

            os.makedirs(os.path.join(OUTPUT_DIR, "img"), exist_ok=True)
            os.makedirs(os.path.join(OUTPUT_DIR, "data"), exist_ok=True)
            
            fig.suptitle(fr"{ID_list[j]}", fontsize=16)
            plt.savefig(os.path.join(OUTPUT_DIR, "img", f"{ID_list[j]}.png"))
            np.save(os.path.join(OUTPUT_DIR, "data", f"{ID_list[j]}.npy"), grand_thickness)

            # plot_individual_layer(grand_thickness, f'{ID_list[j]}', True, f'{ID_list[j]}.png')
    
            # plt.close()

In [ ]:
# Groups to process: each name must match a <name>_C folder in DATA_DIR.
# Add further groups as needed, e.g. ['WT5dpf', 'mutant1', 'mutant2']
conditions = ['WT5dpf']
all_in_one(conditions)

Combine

In [ ]:
# Load the per-fish thickness maps saved above and average them per group.
# Files are matched by name, so each fish ID must contain its group name.
# To add a group, uncomment (or copy) the matching lines below.
paths_WT5dpf = sorted(glob.glob(os.path.join(OUTPUT_DIR, "data", "*WT*")))
#paths_mutant1 = sorted(glob.glob(os.path.join(OUTPUT_DIR, "data", "*mutant1*")))
#paths_mutant2 = sorted(glob.glob(os.path.join(OUTPUT_DIR, "data", "*mutant2*")))
#paths_mutant3 = sorted(glob.glob(os.path.join(OUTPUT_DIR, "data", "*mutant3*")))
#paths_mutant4 = sorted(glob.glob(os.path.join(OUTPUT_DIR, "data", "*mutant4*")))
#paths_mutant5 = sorted(glob.glob(os.path.join(OUTPUT_DIR, "data", "*mutant5*")))
#paths_mutant6 = sorted(glob.glob(os.path.join(OUTPUT_DIR, "data", "*mutant6*")))

WT5dpf_combine = np.zeros((len(paths_WT5dpf),360, 180))
#mutant1_combine = np.zeros((len(paths_mutant1),360, 180))
#mutant2_combine = np.zeros((len(paths_mutant2),360, 180))
#mutant3_combine = np.zeros((len(paths_mutant3),360, 180))
#mutant4_combine = np.zeros((len(paths_mutant4),360, 180))
#mutant5_combine = np.zeros((len(paths_mutant5),360, 180))
#mutant6_combine = np.zeros((len(paths_mutant6),360, 180))

for idx,path in enumerate(paths_WT5dpf):
    WT5dpf_combine[idx] = np.load(path)

#for idx,path in enumerate(paths_mutant1):
    #mutant1_combine[idx] = np.load(path)

#for idx,path in enumerate(paths_mutant2):
    #mutant2_combine[idx] = np.load(path)

#for idx,path in enumerate(paths_mutant3):
    #mutant3_combine[idx] = np.load(path)

#for idx,path in enumerate(paths_mutant4):
    #mutant4_combine[idx] = np.load(path)

#for idx,path in enumerate(paths_mutant5):
    #mutant5_combine[idx] = np.load(path)

#for idx,path in enumerate(paths_mutant6):
    #mutant6_combine[idx] = np.load(path)

# Plot the group-mean polar maps (saved to outputs/polar/ when save = True)
save = True
plot_individual_layer(WT5dpf_combine.mean(axis=0),'WT5dpf', save, 'WT5dpf.png')
#plot_individual_layer(mutant1_combine.mean(axis=0),'mutant1', save, 'mutant1.png')
#plot_individual_layer(mutant2_combine.mean(axis=0),'mutant2', save, 'mutant2.png')
#plot_individual_layer(mutant3_combine.mean(axis=0),'mutant3', save, 'mutant3.png')
#plot_individual_layer(mutant4_combine.mean(axis=0),'mutant4', save, 'mutant4.png')
#plot_individual_layer(mutant5_combine.mean(axis=0),'mutant5', save, 'mutant5.png')
#plot_individual_layer(mutant6_combine.mean(axis=0),'mutant6', save, 'mutant6.png')
